<a href="https://colab.research.google.com/github/sudemeydan/Flight-Delay-Prediction-with-Machine-Learning/blob/main/Phishing_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"


In [ ]:
# Hücre 1: Gerekli kütüphaneler

import pandas as pd
import numpy as np
import re


In [ ]:
# Hücre 2: Veri setini okuma

csv_path = "/content/kategorize_edilmis_veri_seti.csv"  # Dosyayı Colab'e yüklediğinde bu isimle olduğunu varsayıyorum

df = pd.read_csv(csv_path)

# İlk 5 satıra bakalım
df.head()


,sender,receiver,date,subject,body,label,urls,kategori
0,dgdg <>,user2.3@gvc.ceas-challenge.cc,"Wed, 06 Aug 2008 02:34:59 +0100",New revolutionary products to help men,\n\n\n\n\n\nMenÂs best flavored ice cream con...,1,http://www.feelwait.com/,kargo
1,"""HomeOfficeSolutions.com"" <uwuslyamnhtqnwlhuwu...",user6@gvc.ceas-challenge.cc,"Wed, 06 Aug 2008 04:25:42 +0000",Last 2 Days to Save on Knoll & Steelcase!,\n\nHomeOfficeSolutions.com \nThis month's buz...,0,http://homeofficesolutions.chtah.com/a/tBHswq$...,güvenli
2,Richard Fortune <hbgdrat.upzltrl@gmail.com>,pq-mism@yahoogroups.com,"Thu, 07 Aug 2008 16:51:24 +0000","Re: [ie-rant] finally, a hallucinogen with no ...","I prefer cake..\n\nOn Nov 7, 2007 12:30 AM, sk...",0,http://ie.youtube.com/watch?v=Bk7CxHCasts,güvenli
3,Bessie Field <fhusillo@emp.uc3m.es>,user2.4@gvc.ceas-challenge.cc,"Wed, 06 Aug 2008 16:03:38 +0700",Katerina age 29 -on dating,+++++++++++++++++++++++++++++++++++++++\nDatin...,1,http://oupyriihylo.narod.ru/?q=XegTYHm,kargo
4,Elinor Maher <Elinor@bol.net.in>,user5@gvc.ceas-challenge.cc,"Thu, 07 Aug 2008 18:57:20 +0530",Don?t let her leave discontented,Stop complaining about your bad luck in love! ...,1,http://falldear.com/,ödeme/finansal işlem


In [ ]:
# Hücre 3: Genel bilgi

print("Boyut:", df.shape)  # satır, sütun sayısı
print("\nSütunlar:", df.columns.tolist())

print("\n--- info() ---")
print(df.info())

print("\n--- Eksik değer sayıları ---")
print(df.isna().sum())


Boyut: (13052, 8)

Sütunlar: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls', 'kategori']

--- info() ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13052 entries, 0 to 13051
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sender    13052 non-null  object
 1   receiver  12925 non-null  object
 2   date      13052 non-null  object
 3   subject   13050 non-null  object
 4   body      13052 non-null  object
 5   label     13052 non-null  int64 
 6   urls      13052 non-null  object
 7   kategori  13052 non-null  object
dtypes: int64(1), object(7)
memory usage: 815.9+ KB
None

--- Eksik değer sayıları ---
sender        0
receiver    127
date          0
subject       2
body          0
label         0
urls          0
kategori      0
dtype: int64


In [ ]:
# Hücre 4: Etiket dağılımlarına bakış

print("Label dağılımı (0=güvenli, 1=phishing):")
print(df['label'].value_counts())
print("\nOranlar:")
print(df['label'].value_counts(normalize=True))

print("\nKategori dağılımı:")
print(df['kategori'].value_counts())


Label dağılımı (0=güvenli, 1=phishing):
label
1    6670
0    6382
Name: count, dtype: int64

Oranlar:
label
1    0.511033
0    0.488967
Name: proportion, dtype: float64

Kategori dağılımı:
kategori
güvenli                        6382
kargo                          4652
ödeme/finansal işlem           1045
sosyal_medya                    413
e-ticaret                       398
diğer (spam/phishing genel)      81
kurumsal/iş                      60
bankacılık                       21
Name: count, dtype: int64


In [ ]:
# Hücre 5: Subject ve body'yi tek metinde birleştirme

# Eksik subject/body varsa boş string ile dolduralım
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')

# Yeni bir text sütunu oluşturalım
df['text'] = df['subject'] + ' ' + df['body']

# Kontrol amaçlı ilk 3 satıra bakalım
df[['subject', 'body', 'text']].head(3)


,subject,body,text
0,New revolutionary products to help men,\n\n\n\n\n\nMenÂs best flavored ice cream con...,New revolutionary products to help men \n\n\n\...
1,Last 2 Days to Save on Knoll & Steelcase!,\n\nHomeOfficeSolutions.com \nThis month's buz...,Last 2 Days to Save on Knoll & Steelcase! \n\n...
2,"Re: [ie-rant] finally, a hallucinogen with no ...","I prefer cake..\n\nOn Nov 7, 2007 12:30 AM, sk...","Re: [ie-rant] finally, a hallucinogen with no ..."


In [ ]:
# Hücre 6: Metin uzunluklarına bakış

lengths = df['text'].str.len()

print(lengths.describe())
print("Maksimum uzunluk:", lengths.max())


count     13052.000000
mean       2049.679513
std        4503.595393
min          51.000000
25%         268.000000
50%         814.000000
75%        2526.250000
max      144114.000000
Name: text, dtype: float64
Maksimum uzunluk: 144114


In [ ]:
# Hücre 7: Metin temizleme fonksiyonu

def clean_text(s: str) -> str:
    """
    E-posta metnini temizler:
    - URL'leri 'URL' tokenıyla değiştirir
    - E-posta adreslerini 'EMAIL' tokenıyla değiştirir
    - HTML taglerini kaldırır
    - Fazla boşlukları temizler
    - Küçük harfe çevirir
    """
    if not isinstance(s, str):
        s = str(s)

    # Satır sonlarını normalize et
    s = s.replace('\r', ' ')

    # URL'leri işaretle
    s = re.sub(r'http\S+|www\.\S+', ' URL ', s)

    # E-posta adreslerini işaretle
    s = re.sub(r'\S+@\S+', ' EMAIL ', s)

    # HTML taglerini kaldır
    s = re.sub(r'<[^>]+>', ' ', s)

    # Küçük harfe çevir
    s = s.lower()

    # Çoklu boşlukları tek boşluğa indir
    s = re.sub(r'\s+', ' ', s)

    return s.strip()


In [ ]:
# Hücre 8: Temizlenmiş metin sütunu oluşturma

df['clean_text'] = df['text'].apply(clean_text)

# Kontrol için ilk 3 satıra bakalım
df[['text', 'clean_text']].head(3)


,text,clean_text
0,New revolutionary products to help men \n\n\n\...,new revolutionary products to help men menâs ...
1,Last 2 Days to Save on Knoll & Steelcase! \n\n...,last 2 days to save on knoll & steelcase! home...
2,"Re: [ie-rant] finally, a hallucinogen with no ...","re: [ie-rant] finally, a hallucinogen with no ..."


In [ ]:
# Hücre 9: Metinleri maksimum karakter uzunluğuna göre kesme

MAX_CHARS = 5000  # İstersen 3000, 4000 gibi değiştirebilirsin

df['clean_text'] = df['clean_text'].str.slice(0, MAX_CHARS)

# Yeni uzunluk istatistiklerine bakalım
new_lengths = df['clean_text'].str.len()
print(new_lengths.describe())
print("Yeni maksimum uzunluk:", new_lengths.max())


count    13052.000000
mean      1214.488278
std       1332.898079
min         17.000000
25%        234.000000
50%        693.000000
75%       1824.000000
max       5000.000000
Name: clean_text, dtype: float64
Yeni maksimum uzunluk: 5000


In [ ]:
# Hücre 10: Binary phishing etiketi

df['is_phishing'] = df['label'].astype(int)  # 0 veya 1

print(df['is_phishing'].value_counts())


is_phishing
1    6670
0    6382
Name: count, dtype: int64


In [ ]:
# Hücre 11: Kategori etiketlerini sayısal id'lere çevirme

df['kategori'] = df['kategori'].astype('category')
df['kategori_id'] = df['kategori'].cat.codes

# id -> kategori eşleşmesini saklayalım
id2kategori = dict(enumerate(df['kategori'].cat.categories))
print("Kategori id -> isim eşlemesi:")
print(id2kategori)


Kategori id -> isim eşlemesi:
{0: 'bankacılık', 1: 'diğer (spam/phishing genel)', 2: 'e-ticaret', 3: 'güvenli', 4: 'kargo', 5: 'kurumsal/iş', 6: 'sosyal_medya', 7: 'ödeme/finansal işlem'}


In [ ]:
# Hücre 12: Nadir kategorileri tek bir sınıfta toplama (isteğe bağlı)

rare_classes = ['bankacılık', 'kurumsal/iş', 'diğer (spam/phishing genel)']

df['kategori_simplified'] = df['kategori'].apply(
    lambda x: 'diğer_phishing' if x in rare_classes else x
)

df['kategori_simplified'] = df['kategori_simplified'].astype('category')
df['kategori_simplified_id'] = df['kategori_simplified'].cat.codes

print("Basitleştirilmiş kategori dağılımı:")
print(df['kategori_simplified'].value_counts())

simplified_id2kategori = dict(enumerate(df['kategori_simplified'].cat.categories))
print("\nBasitleştirilmiş kategori id -> isim eşlemesi:")
print(simplified_id2kategori)


Basitleştirilmiş kategori dağılımı:
kategori_simplified
güvenli                 6382
kargo                   4652
ödeme/finansal işlem    1045
sosyal_medya             413
e-ticaret                398
diğer_phishing           162
Name: count, dtype: int64

Basitleştirilmiş kategori id -> isim eşlemesi:
{0: 'diğer_phishing', 1: 'e-ticaret', 2: 'güvenli', 3: 'kargo', 4: 'sosyal_medya', 5: 'ödeme/finansal işlem'}


In [ ]:
# Hücre 13: Eğitim ve test setine bölme (binary phishing için)

from sklearn.model_selection import train_test_split

X = df['clean_text'].values
y = df['is_phishing'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Eğitim seti boyutu:", len(X_train))
print("Test seti boyutu:", len(X_test))


Eğitim seti boyutu: 10441
Test seti boyutu: 2611


In [ ]:
# Hücre 15: Kütüphaneleri import et ve model adını belirle

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import numpy as np

# Kullanacağımız model
MODEL_NAME = "distilbert-base-uncased"

# GPU varsa kullanalım (Trainer zaten otomatik kullanıyor ama biz de bakmış olalım)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Kullanılan cihaz:", device)


Kullanılan cihaz: cuda


In [ ]:
# Hücre 16: Tokenizer'ı yükleme

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Hücre 17: E-posta dataset sınıfı

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }
        return item

# Dataset nesnelerini oluşturalım
MAX_LEN = 256

train_dataset = EmailDataset(X_train, y_train, tokenizer, max_length=MAX_LEN)
test_dataset  = EmailDataset(X_test,  y_test,  tokenizer, max_length=MAX_LEN)

print("Eğitim örneği sayısı:", len(train_dataset))
print("Test örneği sayısı:", len(test_dataset))


Eğitim örneği sayısı: 10441
Test örneği sayısı: 2611


In [ ]:
# Hücre 18: DistilBERT tabanlı sınıflandırma modeli

num_labels = 2  # 0 = güvenli, 1 = phishing

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

model.to(device)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
# Hücre 19: Değerlendirme metrikleri

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./phishing_model_binary",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
# Hücre 21: Trainer nesnesi

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,  # küçük projede test'i eval gibi kullanıyoruz
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-3344481035.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Hücre 22: Modeli eğit

train_result = trainer.train()

print("Eğitim tamamlandı.")


Step,Training Loss
100,0.102700
200,0.028500
300,0.023400
400,0.017800
500,0.024900
600,0.017400
700,0.003800
800,0.013200
900,0.004300
1000,0.010200


Eğitim tamamlandı.


In [ ]:
# Hücre 23: Değerlendirme

eval_results = trainer.evaluate()
print("Eval sonuçları:", eval_results)

# Daha detaylı rapor (accuracy, precision, recall, f1 vs.)
pred_output = trainer.predict(test_dataset)
logits = pred_output.predictions
y_pred = np.argmax(logits, axis=-1)

print("\nClassification Report (0=güvenli, 1=phishing):\n")
print(classification_report(y_test, y_pred, digits=4))


Eval sonuçları: {'eval_loss': 0.0052825044840574265, 'eval_accuracy': 0.9992340099578706, 'eval_precision': 1.0, 'eval_recall': 0.9985007496251874, 'eval_f1': 0.9992498124531133, 'eval_runtime': 23.0418, 'eval_samples_per_second': 113.316, 'eval_steps_per_second': 3.559, 'epoch': 3.0}

Classification Report (0=güvenli, 1=phishing):

              precision    recall  f1-score   support

           0     0.9984    1.0000    0.9992      1277
           1     1.0000    0.9985    0.9992      1334

    accuracy                         0.9992      2611
   macro avg     0.9992    0.9993    0.9992      2611
weighted avg     0.9992    0.9992    0.9992      2611



In [ ]:
# Hücre 24: Model ve tokenizer'ı kaydet

save_path = "./phishing_binary_model_distilbert"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model ve tokenizer kaydedildi:", save_path)


Model ve tokenizer kaydedildi: ./phishing_binary_model_distilbert


In [ ]:
# Hücre 25: Nadir kategorileri birleştiriyoruz (eğer önceden yaptıysan tekrar yapma)

rare_classes = ['bankacılık', 'kurumsal/iş', 'diğer (spam/phishing genel)']

df['kategori_simplified'] = df['kategori'].apply(
    lambda x: 'diğer_phishing' if x in rare_classes else x
)

df['kategori_simplified'] = df['kategori_simplified'].astype('category')
df['kategori_simplified_id'] = df['kategori_simplified'].cat.codes
simplified_id2kategori = dict(enumerate(df['kategori_simplified'].cat.categories))

simplified_id2kategori


{0: 'diğer_phishing',
 1: 'e-ticaret',
 2: 'güvenli',
 3: 'kargo',
 4: 'sosyal_medya',
 5: 'ödeme/finansal işlem'}

In [ ]:
# Hücre 26: Kategori eğitim/test bölme

X_cat = df['clean_text'].values
y_cat = df['kategori_simplified_id'].values

from sklearn.model_selection import train_test_split

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cat, y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)

len(Xc_train), len(Xc_test)


(10441, 2611)

In [ ]:
import numpy as np
import torch

# yc_train içinde her sınıftan kaç örnek var?
class_counts = np.bincount(yc_train)
num_classes = len(class_counts)

print("Sınıf sayıları:", class_counts)

# ters frekans ağırlığı (az sınıfa yüksek ağırlık)
weights = 1.0 / class_counts
weights = weights / weights.sum() * num_classes  # normalize

print("Class weights:", weights)

class_weights = torch.tensor(weights, dtype=torch.float).to(device)
class_weights


Sınıf sayıları: [ 130  318 5105 3722  330  836]
Class weights: [2.97229891 1.21509075 0.07569028 0.10381485 1.17090563 0.46219959]


tensor([2.9723, 1.2151, 0.0757, 0.1038, 1.1709, 0.4622], device='cuda:0')

In [ ]:
# kargo sınıfının id'sini bul
kargo_id = [k for k,v in simplified_id2kategori.items() if v=="kargo"][0]
print("kargo_id:", kargo_id)

# kargo ağırlığını %20 daha düşür
class_weights[kargo_id] = class_weights[kargo_id] * 0.8

print("Güncellenmiş class weights:", class_weights)


kargo_id: 3
Güncellenmiş class weights: tensor([2.9723, 1.2151, 0.0757, 0.0831, 1.1709, 0.4622], device='cuda:0')


In [ ]:
# Hücre 27: Multi-class dataset

class CategoryDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

cat_train_ds = CategoryDataset(Xc_train, yc_train, tokenizer)
cat_test_ds  = CategoryDataset(Xc_test,  yc_test,  tokenizer)


In [ ]:
# Hücre 28: Multi-class model oluşturma

num_cat_labels = len(df['kategori_simplified_id'].unique())
num_cat_labels


6

In [ ]:
model_cat = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_cat_labels
).to(device)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Hücre 29

training_args_cat = TrainingArguments(
    output_dir="./phishing_model_category",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    # 🔥 yeni sürümler için num_items_in_batch ve diğer kwarg'ları kabul ediyoruz
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


In [ ]:
# Hücre 30: Multi-class metrikler

from sklearn.metrics import f1_score, accuracy_score

def compute_metrics_cat(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}


In [ ]:
trainer_cat = WeightedTrainer(
    model=model_cat,
    args=training_args_cat,
    train_dataset=cat_train_ds,
    eval_dataset=cat_test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_cat,
    class_weights=class_weights
)


/tmp/ipython-input-1464505176.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [ ]:
# Hücre 31: Kategori modelini eğit

trainer_cat.train()


Step,Training Loss
100,1.295900
200,1.067100
300,0.894400
400,0.956200
500,0.874100
600,0.990400
700,0.817700
800,0.654800
900,0.732000
1000,0.662400


TrainOutput(global_step=1959, training_loss=0.7418668346784747, metrics={'train_runtime': 852.5484, 'train_samples_per_second': 36.74, 'train_steps_per_second': 2.298, 'total_flos': 2074786156956672.0, 'train_loss': 0.7418668346784747, 'epoch': 3.0})

In [ ]:
# Hücre 32: Test değerlendirme

eval_results_cat = trainer_cat.evaluate()
print(eval_results_cat)

pred_output_cat = trainer_cat.predict(cat_test_ds)
logits_cat = pred_output_cat.predictions
y_pred_cat = np.argmax(logits_cat, axis=-1)

print(classification_report(
    yc_test, y_pred_cat,
    target_names=[simplified_id2kategori[i] for i in range(num_cat_labels)],
    digits=4
))


{'eval_loss': 1.0903986692428589, 'eval_accuracy': 0.8441210264266564, 'eval_f1': 0.8545020206715916, 'eval_runtime': 23.3644, 'eval_samples_per_second': 111.751, 'eval_steps_per_second': 3.51, 'epoch': 3.0}
                      precision    recall  f1-score   support

      diğer_phishing     0.2273    0.3125    0.2632        32
           e-ticaret     0.2640    0.4125    0.3220        80
             güvenli     0.9984    0.9969    0.9976      1277
               kargo     0.8745    0.7269    0.7939       930
        sosyal_medya     0.3356    0.5904    0.4279        83
ödeme/finansal işlem     0.6573    0.7799    0.7133       209

            accuracy                         0.8441      2611
           macro avg     0.5595    0.6365    0.5863      2611
        weighted avg     0.8740    0.8441    0.8545      2611



In [ ]:
# Hücre 33: Binary ve kategori modellerini kaydet

binary_model_path = "./phishing_binary_model_distilbert"
category_model_path = "./phishing_category_model_distilbert"

# Binary model ve tokenizer
model.save_pretrained(binary_model_path)
tokenizer.save_pretrained(binary_model_path)

# Kategori model ve tokenizer
model_cat.save_pretrained(category_model_path)
tokenizer.save_pretrained(category_model_path)

print("Kaydedilen klasörler:")
print("Binary model:", binary_model_path)
print("Kategori model:", category_model_path)


Kaydedilen klasörler:
Binary model: ./phishing_binary_model_distilbert
Kategori model: ./phishing_category_model_distilbert


In [ ]:
# Hücre 34: Inference için modelleri ve tokenizer'ı yükle

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Inference cihazı:", device)

binary_model_path = "./phishing_binary_model_distilbert"
category_model_path = "./phishing_category_model_distilbert"

tokenizer = AutoTokenizer.from_pretrained(binary_model_path)

binary_model = AutoModelForSequenceClassification.from_pretrained(binary_model_path).to(device)
category_model = AutoModelForSequenceClassification.from_pretrained(category_model_path).to(device)

binary_model.eval()
category_model.eval()

print("Modeller yüklendi.")


Inference cihazı: cuda
Modeller yüklendi.


In [ ]:
# Hücre 35: Metin temizleme (eğitimdekiyle aynı)

import re

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        s = str(s)

    s = s.replace('\r', ' ')
    s = re.sub(r'http\S+|www\.\S+', ' URL ', s)
    s = re.sub(r'\S+@\S+', ' EMAIL ', s)
    s = re.sub(r'<[^>]+>', ' ', s)
    s = s.lower()
    s = re.sub(r'\s+', ' ', s)

    return s.strip()


In [ ]:
# Hücre 36: Basitleştirilmiş kategori id -> isim sözlüğünü tekrar oluştur (gerekirse)

df['kategori_simplified'] = df['kategori_simplified'].astype('category')
simplified_id2kategori = dict(enumerate(df['kategori_simplified'].cat.categories))

simplified_id2kategori


{0: 'diğer_phishing',
 1: 'e-ticaret',
 2: 'güvenli',
 3: 'kargo',
 4: 'sosyal_medya',
 5: 'ödeme/finansal işlem'}

In [ ]:
# Hücre 37: Yardımcı tahmin fonksiyonları

import torch.nn.functional as F

def predict_phishing(text: str):
    """
    Girdi metinden:
    - phishing olup olmadığını,
    - sınıf olasılıklarını döndürür.
    """
    clean = clean_text(text)

    enc = tokenizer(
        clean,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = binary_model(**enc)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).cpu().numpy().flatten()

    # Etiketler: 0 = güvenli, 1 = phishing (eğitimde böyleydi)
    prob_safe = float(probs[0])
    prob_phish = float(probs[1])

    predicted_label = 1 if prob_phish >= prob_safe else 0
    label_name = "phishing" if predicted_label == 1 else "güvenli"

    return label_name, prob_safe, prob_phish


def predict_category(text: str):
    """
    Girdi metinden:
    - kategori id,
    - kategori ismi,
    - tüm kategoriler için olasılık dağılımı döndürür.
    """
    clean = clean_text(text)

    enc = tokenizer(
        clean,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = category_model(**enc)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).cpu().numpy().flatten()

    pred_id = int(np.argmax(probs))
    pred_name = simplified_id2kategori[pred_id]

    # kategori adı -> olasılık sözlüğü (Gradio Label için)
    prob_dict = {
        simplified_id2kategori[i]: float(probs[i])
        for i in range(len(probs))
    }

    return pred_id, pred_name, prob_dict


In [ ]:
# Hücre 38: Açıklama (explanation) üretici fonksiyon

def explain_prediction(text: str, phishing_label: str, category_name: str):
    """
    Metin içindeki basit pattern'lere bakarak neden phishing / değil açıklaması üretir.
    """
    clean = clean_text(text)
    lower = clean.lower()

    reasons = []

    # 1) Aciliyet ifadeleri
    urgency_phrases = [
        "urgent", "immediately", "as soon as possible",
        "hemen", "acil", "derhal", "24 saat içinde"
    ]
    if any(p in lower for p in urgency_phrases):
        reasons.append("Mesajda alıcıyı aceleye getirmeye çalışan aciliyet ifadeleri kullanılmış.")

    # 2) Hesap / şifre / doğrulama isteme
    credential_phrases = [
        "verify your account", "login", "password", "credentials",
        "hesabınızı doğrulayın", "şifrenizi girin", "giriş yapın", "kimlik doğrulama"
    ]
    if any(p in lower for p in credential_phrases):
        reasons.append("Hesap bilgisi veya şifre doğrulama talebi içeriyor.")

    # 3) Finansal / ödeme / para
    financial_phrases = [
        "payment", "invoice", "transaction", "bank",
        "ödeme", "fatura", "banka", "hesap ekstresi"
    ]
    if any(p in lower for p in financial_phrases):
        reasons.append("Finansal işlem, ödeme veya banka ile ilgili ifadeler kullanılmış.")

    # 4) Kargo / teslimat
    shipping_phrases = [
        "shipment", "cargo", "tracking number", "delivered",
        "kargo", "teslimat", "gönderi", "takip numarası"
    ]
    if any(p in lower for p in shipping_phrases):
        reasons.append("Kargo / teslimat ile ilgili içerik barındırıyor.")

    # 5) Ödül / çekiliş / kazandınız
    reward_phrases = [
        "you won", "congratulations", "prize", "lottery",
        "kazandınız", "tebrikler", "ödül", "çekiliş"
    ]
    if any(p in lower for p in reward_phrases):
        reasons.append("Ödül / çekiliş / kazandınız tarzı ifadeler içeriyor.")

    # 6) URL var mı?
    if "url" in lower or "http" in text.lower() or "www." in text.lower():
        reasons.append("Mesajda tıklanması istenen en az bir bağlantı (URL) bulunuyor.")

    # 7) Şüpheli alan adları
    suspicious_domains = [".ru", ".cn", ".xyz", "bit.ly", "tinyurl", ".top"]
    if any(dom in lower for dom in suspicious_domains):
        reasons.append("Şüpheli veya kısaltılmış alan adları içeriyor olabilir.")

    # 8) Çok fazla ünlem veya büyük harf
    if text.count("!") >= 3:
        reasons.append("Birden fazla ünlem işareti kullanılmış, bu da duygusal baskı işareti olabilir.")
    if any(word.isupper() and len(word) >= 4 for word in text.split()):
        reasons.append("Tamamen büyük harfle yazılmış kelimeler bulunuyor (dikkat çekme / baskı amacı taşıyabilir).")

    # Şimdi phishing / güvenli durumuna göre açıklamayı farklı kur
    if phishing_label == "phishing":
        if not reasons:
            reasons.append("Metin genel yapısı ve kelime seçimi phishing maillerine benziyor.")
        header = "Bu mesajın phishing olma ihtimali yüksek çünkü:"
    else:
        if not reasons:
            reasons.append("Mesajda tipik phishing unsurlarına (aciliyet, şifre isteme, şüpheli link vb.) güçlü bir örnek gözükmüyor.")
        header = "Bu mesaj genel olarak güvenli görünüyor, çünkü:"

    # Kategori bilgisini de ekleyelim
    if category_name != "":
        reasons.append(f"İçerik genel olarak '{category_name}' kategorisine benziyor.")

    explanation = header + "\n- " + "\n- ".join(reasons)
    return explanation


In [ ]:
# Hücre 40: Arayüzde kullanılacak ana fonksiyon

import gradio as gr

def analyze_email(subject, body):
    # None gelirse boş string yap
    subject = subject or ""
    body = body or ""
    full_text = subject + " " + body

    # 1) Phishing tahmini
    phishing_label, prob_safe, prob_phish = predict_phishing(full_text)

    # Gradio Label için dict (class -> probability)
    phishing_probs_dict = {
        "güvenli": prob_safe,
        "phishing": prob_phish
    }

    # 2) Kategori tahmini
    cat_id, cat_name, cat_probs_dict = predict_category(full_text)

    # 3) Açıklama üret
    explanation = explain_prediction(full_text, phishing_label, cat_name)

    # Sonuçları döndür
    return phishing_probs_dict, cat_probs_dict, explanation


In [ ]:
# Hücre 41: Gradio arayüzü

phishing_label_component = gr.Label(label="Phishing Tahmini (olasılıklar)")
category_label_component = gr.Label(label="Kategori Tahmini (olasılıklar)")

iface = gr.Interface(
    fn=analyze_email,
    inputs=[
        gr.Textbox(label="Konu (Subject)", lines=2, placeholder="Örn: Ödemeniz hakkında önemli bilgilendirme"),
        gr.Textbox(label="Mesaj İçeriği (Body)", lines=10, placeholder="E-posta gövdesini buraya yapıştırın...")
    ],
    outputs=[
        phishing_label_component,
        category_label_component,
        gr.Textbox(label="Açıklama", lines=8)
    ],
    title="Phishing Tespit Sistemi (LLM tabanlı)",
    description=(
        "Bu arayüz, eğitilmiş DistilBERT tabanlı modeller ile e-postanın phishing olup olmadığını "
        "ve hangi kategoriye (kargo, ödeme, sosyal medya vb.) daha çok benzediğini tahmin eder. "
        "Ayrıca kararın nedenini Türkçe olarak açıklar."
    )
)

iface.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f132f90660261eb881.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f132f90660261eb881.gradio.live
